# CUDA Source Notebook

### Setup: `uv`

In [64]:
%%bash
uv self update
uv sync

info: Checking for updates...


success: You're already on version v0.13.1 of uv (the latest version).
Resolved 89 packages in 1ms
Checked 84 packages in 1ms


### Setup: Cleanup Files
- Delete every generated file so the notebook rebuilds from scratch.

In [65]:
%%bash
cleanup=(
    *.c
    *.h
    *.cpp
    *.hpp
    *.cu
    *.cuh
    *.o
    *.so
    *.py
    __pycache__
)
for file in "${cleanup[@]}"; do
    rm -rf "$file"
done

### Setup: `nvidia-smi`

In [66]:
%%bash
nvidia-smi

Fri Oct 16 03:42:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.57.01              KMD Version: 610.88        CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        On  |   00000000:0C:00.0 Off |                  N/A |
|  0%   42C    P8             23W /  350W |    1197MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### CUDA Kernel: `hello.cu`

In [67]:
%%writefile hello.cu
#include <cstdint>
#include <cstdio>
#include <cuda_runtime.h>

__global__
static void hello_kernel() {
  printf("Hello, World!\n");
}

extern "C" {
  cudaError_t hello() {
    constexpr uint32_t grid_size = 1;
    constexpr uint32_t block_size = 1;

    hello_kernel<<<grid_size, block_size>>>();
    auto kernel_err = cudaGetLastError();
    if (kernel_err != cudaSuccess) {
      return kernel_err;
    }

    return cudaDeviceSynchronize();
  }

  const char* get_error_string(cudaError_t err) {
    return cudaGetErrorString(err);
  }
}

Writing hello.cu


In [68]:
%%bash
nvcc -std=c++23 -arch=native -O3 --compiler-options '-fPIC' -shared hello.cu -o lib_hello.so

### CUDA Kernel: `hello.py`

In [69]:
%%writefile hello.py
import sys
from ctypes import (
    CDLL,
    c_char_p,
    c_int,
    c_uint32,
)
from pathlib import Path

lib_name = 'lib_hello.so'
try:
    lib = CDLL(str(Path(__file__).parent / lib_name))
except OSError as e:
    raise RuntimeError(f"Failed to load {lib_name}") from e

lib.hello.argtypes = []
lib.hello.restype  = c_int

lib.get_error_string.argtypes = [c_int]
lib.get_error_string.restype  = c_char_p

def get_error_string(err: int) -> str:
    return lib.get_error_string(err).decode('utf-8')

def hello() -> None:
    if (err := lib.hello()) != 0:
        raise RuntimeError(f"Kernel failed: {get_error_string(err)}")

if __name__ == '__main__':
    if len(sys.argv) != 1:
        print(f"Usage: {sys.argv[0]}", file=sys.stderr)
        sys.exit(1)

    hello()

Writing hello.py


In [70]:
%%bash
python hello.py

Hello, World!


### Device Properties: `device_properties.cu`
- Ask the runtime what the GPU can do with `cudaGetDeviceProperties`; no kernel yet.

In [71]:
%%writefile device_properties.cu
#include <cstdint>
#include <cstdio>
#include <cuda_runtime.h>

extern "C" {
  cudaError_t device_properties() {
    constexpr uint32_t device_id = 0;

    cudaDeviceProp prop{};
    auto prop_err = cudaGetDeviceProperties(&prop, device_id);
    if (prop_err != cudaSuccess) {
      return prop_err;
    }

    printf("Name:                  %s\n", prop.name);
    printf("Compute capability:    %d.%d\n", prop.major, prop.minor);
    printf("Multiprocessors:       %d\n", prop.multiProcessorCount);
    printf("Global memory:         %zu bytes\n", prop.totalGlobalMem);
    printf("Warp size:             %d\n", prop.warpSize);
    printf("Max threads per block: %d\n", prop.maxThreadsPerBlock);
    printf("Max block size:        %d x %d x %d\n", prop.maxThreadsDim[0], prop.maxThreadsDim[1], prop.maxThreadsDim[2]);
    printf("Max grid size:         %d x %d x %d\n", prop.maxGridSize[0], prop.maxGridSize[1], prop.maxGridSize[2]);

    return cudaSuccess;
  }

  const char* get_error_string(cudaError_t err) {
    return cudaGetErrorString(err);
  }
}

Writing device_properties.cu


In [72]:
%%bash
nvcc -std=c++23 -arch=native -O3 --compiler-options '-fPIC' -shared device_properties.cu -o lib_device_properties.so

### Device Properties: `device_properties.py`

In [73]:
%%writefile device_properties.py
import sys
from ctypes import (
    CDLL,
    c_char_p,
    c_int,
)
from pathlib import Path

lib_name = 'lib_device_properties.so'
try:
    lib = CDLL(str(Path(__file__).parent / lib_name))
except OSError as e:
    raise RuntimeError(f"Failed to load {lib_name}") from e

lib.device_properties.argtypes = []
lib.device_properties.restype  = c_int

lib.get_error_string.argtypes = [c_int]
lib.get_error_string.restype  = c_char_p

def get_error_string(err: int) -> str:
    return lib.get_error_string(err).decode('utf-8')

def device_properties() -> None:
    if (err := lib.device_properties()) != 0:
        raise RuntimeError(f"Device properties failed: {get_error_string(err)}")

if __name__ == '__main__':
    if len(sys.argv) != 1:
        print(f"Usage: {sys.argv[0]}", file=sys.stderr)
        sys.exit(1)

    device_properties()

Writing device_properties.py


In [74]:
%%bash
python device_properties.py

Name:                  NVIDIA GeForce RTX 3090
Compute capability:    8.6
Multiprocessors:       82
Global memory:         25769279488 bytes
Warp size:             32
Max threads per block: 1024
Max block size:        1024 x 1024 x 64
Max grid size:         2147483647 x 65535 x 65535


### Bandwidth: `cuda_ptr.cuh`
- Owning pointers for pinned host memory, device memory and events: each frees itself when it goes out of scope, so an early `return` never leaks.

In [75]:
%%writefile cuda_ptr.cuh
#pragma once
#include <memory>
#include <type_traits>
#include <cuda_runtime.h>

struct host_deleter {
  void operator()(void* ptr) const {
    cudaFreeHost(ptr);
  }
};

struct device_deleter {
  void operator()(void* ptr) const {
    cudaFree(ptr);
  }
};

struct event_deleter {
  void operator()(cudaEvent_t event) const {
    cudaEventDestroy(event);
  }
};

template <typename T>
using host_ptr = std::unique_ptr<T, host_deleter>;

template <typename T>
using device_ptr = std::unique_ptr<T, device_deleter>;

using event_ptr = std::unique_ptr<std::remove_pointer_t<cudaEvent_t>, event_deleter>;

Writing cuda_ptr.cuh


### Bandwidth: `bandwidth.cu`
- Allocate GPU memory with `cudaMalloc`, copy with `cudaMemcpy`, and time each copy with CUDA events.
- The host buffer is pinned (`cudaMallocHost`), which lets the GPU copy at full PCIe speed.

In [76]:
%%writefile bandwidth.cu
#include <cstdint>
#include <cuda_runtime.h>
#include "cuda_ptr.cuh"

static cudaError_t time_copy(
  float* elapsed_ms, 
  void* dst, 
  const void* src,
  cudaEvent_t start,
  cudaEvent_t stop,
  uint64_t size,
  cudaMemcpyKind kind)
{
  // trigger the start event
  auto start_err = cudaEventRecord(start);
  if (start_err != cudaSuccess) {
    return start_err;
  }

  // copy data
  auto copy_err = cudaMemcpy(dst, src, size, kind);
  if (copy_err != cudaSuccess) {
    return copy_err;
  }

  // trigger the stop event
  auto stop_err = cudaEventRecord(stop);
  if (stop_err != cudaSuccess) {
    return stop_err;
  }

  // wait for the stop event
  auto sync_err = cudaEventSynchronize(stop);
  if (sync_err != cudaSuccess) {
    return sync_err;
  }

  // read the time between the events
  auto elapsed_err = cudaEventElapsedTime(elapsed_ms, start, stop);
  if (elapsed_err != cudaSuccess) {
    return elapsed_err;
  }

  return cudaSuccess;
}

extern "C" {
  cudaError_t bandwidth(
    float* host_to_host_ms, 
    float* host_to_device_ms, 
    float* device_to_host_ms, 
    float* device_to_device_ms, 
    uint64_t size) 
  {
    // allocate the pinned source host buffer
    uint8_t* src_host_raw{};
    auto src_host_malloc_err = cudaMallocHost(&src_host_raw, size);
    if (src_host_malloc_err != cudaSuccess) {
      return src_host_malloc_err;
    }
    host_ptr<uint8_t> src_host(src_host_raw);

    // allocate the pinned destination host buffer
    uint8_t* dst_host_raw{};
    auto dst_host_malloc_err = cudaMallocHost(&dst_host_raw, size);
    if (dst_host_malloc_err != cudaSuccess) {
      return dst_host_malloc_err;
    }
    host_ptr<uint8_t> dst_host(dst_host_raw);

    // allocate the source device buffer
    uint8_t* src_device_raw{};
    auto src_device_malloc_err = cudaMalloc(&src_device_raw, size);
    if (src_device_malloc_err != cudaSuccess) {
      return src_device_malloc_err;
    }
    device_ptr<uint8_t> src_device(src_device_raw);

    // allocate the destination device buffer
    uint8_t* dst_device_raw{};
    auto dst_device_malloc_err = cudaMalloc(&dst_device_raw, size);
    if (dst_device_malloc_err != cudaSuccess) {
      return dst_device_malloc_err;
    }
    device_ptr<uint8_t> dst_device(dst_device_raw);

    // create the start event
    cudaEvent_t start_raw{};
    auto start_create_err = cudaEventCreate(&start_raw);
    if (start_create_err != cudaSuccess) {
      return start_create_err;
    }
    event_ptr start(start_raw);

    // create the stop event
    cudaEvent_t stop_raw{};
    auto stop_create_err = cudaEventCreate(&stop_raw);
    if (stop_create_err != cudaSuccess) {
      return stop_create_err;
    }
    event_ptr stop(stop_raw);

    // time host to host
    auto host_to_host_err = time_copy(host_to_host_ms, dst_host.get(), src_host.get(), start.get(), stop.get(), size, cudaMemcpyHostToHost);
    if (host_to_host_err != cudaSuccess) {
      return host_to_host_err;
    }

    // time host to device
    auto host_to_device_err = time_copy(host_to_device_ms, dst_device.get(), src_host.get(), start.get(), stop.get(), size, cudaMemcpyHostToDevice);
    if (host_to_device_err != cudaSuccess) {
      return host_to_device_err;
    }

    // time device to host
    auto device_to_host_err = time_copy(device_to_host_ms, dst_host.get(), src_device.get(), start.get(), stop.get(), size, cudaMemcpyDeviceToHost);
    if (device_to_host_err != cudaSuccess) {
      return device_to_host_err;
    }

    // time device to device
    auto device_to_device_err = time_copy(device_to_device_ms, dst_device.get(), src_device.get(), start.get(), stop.get(), size, cudaMemcpyDeviceToDevice);
    if (device_to_device_err != cudaSuccess) {
      return device_to_device_err;
    }

    return cudaSuccess;
  }

  const char* get_error_string(cudaError_t err) {
    return cudaGetErrorString(err);
  }
}

Writing bandwidth.cu


In [77]:
%%bash
nvcc -std=c++23 -arch=native -O3 --compiler-options '-fPIC' -shared bandwidth.cu -o lib_bandwidth.so

### Bandwidth: `bandwidth.py`
- Convert the times to GB/s; host-to-host and device-to-device copies read and write the same memory, so they count every byte twice.

In [78]:
%%writefile bandwidth.py
import sys
from ctypes import (
    CDLL,
    POINTER,
    byref,
    c_char_p,
    c_float,
    c_int,
    c_uint64,
)
from pathlib import Path

lib_name = 'lib_bandwidth.so'
try:
    lib = CDLL(str(Path(__file__).parent / lib_name))
except OSError as e:
    raise RuntimeError(f"Failed to load {lib_name}") from e

lib.bandwidth.argtypes = [POINTER(c_float), POINTER(c_float), POINTER(c_float), POINTER(c_float), c_uint64]
lib.bandwidth.restype  = c_int

lib.get_error_string.argtypes = [c_int]
lib.get_error_string.restype  = c_char_p

bytes_per_mib = 2**20
bytes_per_gb = 10**9
ms_per_s = 10**3

def get_error_string(err: int) -> str:
    return lib.get_error_string(err).decode('utf-8')

def gb_per_s(size: int, ms: float) -> float:
    return size / bytes_per_gb / (ms / ms_per_s)

def bandwidth(size: int) -> tuple[float, float, float, float]:
    host_to_host_ms = c_float()
    host_to_device_ms = c_float()
    device_to_host_ms = c_float()
    device_to_device_ms = c_float()
    if (err := lib.bandwidth(byref(host_to_host_ms), byref(host_to_device_ms), byref(device_to_host_ms), byref(device_to_device_ms), size)) != 0:
        raise RuntimeError(f"Bandwidth failed: {get_error_string(err)}")
    return (
        gb_per_s(2 * size, host_to_host_ms.value),
        gb_per_s(size, host_to_device_ms.value),
        gb_per_s(size, device_to_host_ms.value),
        gb_per_s(2 * size, device_to_device_ms.value),
    )

if __name__ == '__main__':
    if len(sys.argv) != 2:
        print(f"Usage: {sys.argv[0]} <size_mib>", file=sys.stderr)
        sys.exit(1)

    host_to_host, host_to_device, device_to_host, device_to_device = bandwidth(int(sys.argv[1]) * bytes_per_mib)
    print(f"Host to host:     {host_to_host:7.1f} GB/s")
    print(f"Host to device:   {host_to_device:7.1f} GB/s")
    print(f"Device to host:   {device_to_host:7.1f} GB/s")
    print(f"Device to device: {device_to_device:7.1f} GB/s")

Writing bandwidth.py


In [83]:
%%bash
python bandwidth.py 1024

Host to host:        33.3 GB/s
Host to device:      26.0 GB/s
Device to host:      26.2 GB/s
Device to device:   812.7 GB/s
